In [58]:
import torch

import time
import triton
import triton.language as tl
import torch.nn.functional as F


In [73]:
@triton.jit
def matmul_kernel(X, Y, Z, Out, 
                  stride_xb, stride_xm, stride_xk, 
                  stride_yk, stride_yn,
                  stride_zk, stride_zn,
                  stride_ob, stride_om, stride_on, 
                  B: tl.constexpr, M: tl.constexpr, K: tl.constexpr, 
                  N: tl.constexpr, BLOCK_I: tl.constexpr,
                  BLOCK_J: tl.constexpr, BLOCK_K: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    off_batch = tl.program_id(2)
    
    x_batch_offset = off_batch * stride_xb
    # y_batch_offset = off_batch * stride_yb
    o_batch_offset = off_batch * stride_ob
    
    X_block_ptr = tl.make_block_ptr(
        base=X + x_batch_offset,
        shape=(M, K),
        strides=(stride_xm, stride_xk),
        offsets=(pid_m * BLOCK_I, 0),
        block_shape=(BLOCK_I, BLOCK_K),
        order=(1, 0)
    )
    
    Y_block_ptr = tl.make_block_ptr(
        base=Y, #+ y_batch_offset,
        shape=(K, N),
        strides=(stride_yk, stride_yn),
        offsets=(0, pid_n * BLOCK_J),
        block_shape=(BLOCK_K, BLOCK_J),
        order=(1, 0)
    )

    Z_block_ptr = tl.make_block_ptr(
        base=Z, #+ y_batch_offset,
        shape=(K, N),
        strides=(stride_zk, stride_zn),
        offsets=(0, pid_n * BLOCK_J),
        block_shape=(BLOCK_K, BLOCK_J),
        order=(1, 0)
    )
    
    O_block_ptr = tl.make_block_ptr(
        base=Out + o_batch_offset,
        shape=(M, N),
        strides=(stride_om, stride_on),
        offsets=(pid_m * BLOCK_I, pid_n * BLOCK_J),
        block_shape=(BLOCK_I, BLOCK_J),
        order=(1, 0)
    )
    
    acc_y = tl.zeros((BLOCK_I, BLOCK_J), dtype=tl.float32)
    acc_z = tl.zeros((BLOCK_I, BLOCK_J), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        x = tl.load(X_block_ptr)
        y = tl.load(Y_block_ptr)
        z = tl.load(Z_block_ptr)
        acc_y += tl.dot(x, y)
        acc_z += tl.dot(x, z)
        X_block_ptr = tl.advance(X_block_ptr, (0, BLOCK_K))
        Y_block_ptr = tl.advance(Y_block_ptr, (BLOCK_K, 0))
        Z_block_ptr = tl.advance(Z_block_ptr, (BLOCK_K, 0))
    acc_y = acc_y * tl.sigmoid(acc_y)
    acc = acc_y * acc_z
    tl.store(O_block_ptr, acc)


In [74]:
def matmul(x: torch.Tensor, y: torch.Tensor, z: torch.Tensor):
    assert x.shape[2] == y.shape[0] == z.shape[0]
    b, m, k = x.shape
    n = y.shape[-1]
    output = torch.empty((b, m, n), dtype=torch.float32, device='cuda')
    BLOCK_I=16
    BLOCK_J=16
    BLOCK_K=128
    assert x.is_cuda and y.is_cuda and output.is_cuda
    grid = lambda META: (triton.cdiv(m, META['BLOCK_I']), triton.cdiv(n, META['BLOCK_J']), b)
    print(grid({"BLOCK_I":BLOCK_I, "BLOCK_J":BLOCK_J}))
    matmul_kernel[grid](
        x, y, z, output, 
        x.stride(0), x.stride(1), x.stride(2), 
        y.stride(0), y.stride(1),
        z.stride(0), z.stride(1),
        output.stride(0), output.stride(1), output.stride(2), 
        b, m, k, n, BLOCK_I=BLOCK_I, BLOCK_J=BLOCK_J, BLOCK_K=BLOCK_K
    )
    return output

In [76]:
# torch.manual_seed(0)
i = torch.rand(1, 16, 4096, device='cuda', dtype=torch.float32).contiguous()*0.2
w1 = torch.rand(14336, 4096, device='cuda', dtype=torch.float32).contiguous()*0.2
w3 = torch.rand(14336, 4096, device='cuda', dtype=torch.float32).contiguous()*0.2

st = time.time()
output_torch = F.silu(i @ w1.T) * (i @ w3.T)
print("torch: ", time.time()-st)
torch.cuda.empty_cache()
st = time.time()
output_triton = matmul(i, w1.T, w3.T)
print("triton: ", time.time()-st)

# print(output_torch)
# print(output_triton)



torch:  0.0038802623748779297
(1, 896, 1)
triton:  0.0005903244018554688


In [1]:
output_triton

NameError: name 'output_triton' is not defined

In [78]:
output_torch

tensor([[[1653.1680, 1684.8395, 1665.6807,  ..., 1651.2394, 1698.9885,
          1695.3666],
         [1711.6724, 1717.4836, 1705.6262,  ..., 1699.4518, 1716.0419,
          1735.0029],
         [1651.5242, 1670.7537, 1647.6616,  ..., 1658.7627, 1704.1057,
          1689.3391],
         ...,
         [1657.6248, 1714.1813, 1704.8344,  ..., 1632.9462, 1720.3405,
          1754.7744],
         [1713.8123, 1746.6327, 1723.3066,  ..., 1697.4419, 1743.6998,
          1759.6149],
         [1672.0090, 1698.2775, 1705.0123,  ..., 1669.3789, 1702.2653,
          1721.5825]]], device='cuda:0')